### Cell 01 - Import Dependencies and Basic Settings

This cell imports libraries for numerical computing, plotting, table I/O, and GP-HT-related routines, and sets basic paths or plotting styles used later.

- Purpose: DCT-GPHT unbounded processing of experimental impedance data, dual-branch export, comparison of four condition types, and result saving.


In [ ]:
from __future__ import annotations
import hashlib
import math
import re
import warnings
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.linalg import cho_solve, solve_triangular
from scipy.optimize import minimize
# GP_hilbert_unbounded.py is imported from the current folder. It is based on the companion code of the original GP-HT work by Ciucci et al. (J. Electrochem. Soc. 2020, DOI: 10.1149/1945-7111/aba9c0).
import GP_hilbert_unbounded as gpht_unbounded
warnings.filterwarnings("ignore", category=RuntimeWarning)


### Cell 02 - Experimental Parameter Configuration

This cell configures the input file, output directory, noise levels, sparse-sampling ratios, frequency-limitation/truncation settings, prediction-frequency range, and random seed.

- Purpose: DCT-GPHT unbounded processing of experimental impedance data, dual-branch export, comparison of four condition types, and result saving.
- Main parameters: INPUT_EXCEL, SHEET_NAME_PATTERNS, MAX_SHEETS, SKIP_INVALID_SHEETS, OUTPUT_ROOT, RUN_TIMESTAMP, RUN_ID, DATA_DIR, FIG_DIR, RUN_BRANCHES, INCLUDE_RAW_ANALYSIS, NOISE_LEVELS, SPARSE_RATIOS, LIMITED_PERCENT_LIST, LIMITED_RANGE_LIST, RANDOM_SEED, PRED_FREQ_MIN, PRED_FREQ_MAX.These variables control input/output paths, experimental conditions, frequency ranges, or plotting behavior.


In [ ]:
# User configuration
NOTEBOOK_DIR = Path.cwd()
CASE_DATA_DIR = NOTEBOOK_DIR.parent / "6 Case Data" if (NOTEBOOK_DIR.parent / "6 Case Data").exists() else NOTEBOOK_DIR / "6 Case Data"  # This is the author's local input/output path; please update it before running.
DATASET_NAME = "potato"  # Options: "cell", "potato", "mouse"。
CASE_DATA_FILES = {
    "cell": CASE_DATA_DIR / "case_data_cell.xlsx",
    "potato": CASE_DATA_DIR / "case_data_potato.xlsx",
    "mouse": CASE_DATA_DIR / "case_data_mouse.xlsx",
}
INPUT_EXCEL = CASE_DATA_FILES[DATASET_NAME]  # This is the author's local input/output path; please update it before running.
SHEET_NAME_PATTERNS: list[str] = []  # An empty list reads all sheets; fill in keywords only when name-based filtering is needed.
MAX_SHEETS: int | None = None  # Set to 1 for debugging; keep as None for full runs.
SKIP_INVALID_SHEETS = True  # When all sheets are read, sheets whose first three columns are not impedance data are skipped automatically.
OUTPUT_ROOT = Path(r"C:/Users/CYJ/Desktop/DCT-GPHT_experimental_data_outputs")  # This is the author's local input/output path; please update it before running.
RUN_TIMESTAMP = datetime.now().strftime("%y%m%d%H%M")
RUN_ID = RUN_TIMESTAMP
while (OUTPUT_ROOT / "data" / RUN_ID).exists() or (OUTPUT_ROOT / "figures" / RUN_ID).exists():
    suffix = int(RUN_ID.rsplit("_", 1)[-1]) + 1 if "_" in RUN_ID else 2
    RUN_ID = f"{RUN_TIMESTAMP}_{suffix:02d}"
DATA_DIR = OUTPUT_ROOT / "data" / RUN_ID  # This is the author's local input/output path; please update it before running.
FIG_DIR = OUTPUT_ROOT / "figures" / RUN_ID  # This is the author's local input/output path; please update it before running.
# Options: ["imInput"], ["reInput"], or ["imInput", "reInput"].
RUN_BRANCHES = ["imInput", "reInput"]  # Select imInput, reInput, or both branches.
# Degraded-condition settings; each list can be empty or contain one or more values.
INCLUDE_RAW_ANALYSIS = True
NOISE_LEVELS = [0.10]  # Multiplicative relative noise levels to generate; 0.10 means Z_new = Z * (1 + N(0, 0.10)).
SPARSE_RATIOS = [3]  # Sparse-sampling ratio; 3 means keeping one point every three frequency points.
LIMITED_PERCENT_LIST: list[float] = [0.20]  # Point-count truncation settings; 0.20 removes 10% of points from each low- and high-frequency end.
LIMITED_RANGE_LIST: list[tuple[float, float]] = []  # Frequency-range retention settings; each tuple is (minimum frequency, maximum frequency), and an empty list disables this condition.
RANDOM_SEED = 42
PRED_FREQ_MIN = 100.0
PRED_FREQ_MAX = 1.0e7
N_PRED = 200
ADMITTANCE_SCALE = 1000.0  # Admittance scaling factor; DCT is computed in the admittance domain, and scaling improves numerical stability.
TAU_MIN: float | None = None  # Lower bound of the DCT integral; None estimates it automatically from the current maximum frequency.
# Optimization and uncertainty settings.
OPT_MAXITER = 80  # Maximum number of iterations for the hyperparameter optimizer.
MIN_POINTS = 8  # Minimum number of valid frequency points required for analyzing one spectrum.
MC_SAMPLES = 2000  # Number of Monte Carlo samples used to transform the admittance posterior into the impedance domain.
CI_K_FOR_PLOT = 2  # Standard-deviation multiplier for credible intervals in plots.
NYQUIST_ERR_STRIDE = 12  # Sampling stride for error bars in the Nyquist plot; avoids overly dense error bars.
FIG_DPI = 100  # Resolution used when saving PNG figures.
APPLY_REINPUT_IMAG_OFFSET = False  # Whether to apply an additional constant offset correction to the imaginary component predicted by reInput; disabled by default to preserve the Hilbert-prediction form.
# Automatic tuning settings: all conditions share the same candidate observation-noise floors, rather than using hard-coded raw/noisy/sparse/limited branches.
# The score combines input-side regression error normalized by admittance magnitude with curve roughness.
AUTO_TUNE_SIGMA_N_FLOOR = True  # Whether to run a small candidate search for the observation-noise floor.
SIGMA_N_FLOOR_FACTORS = [1e-8, 1e-3, 1e-2, 3e-2, 1e-1, 3e-1]  # Candidate observation-noise floor ratios used for automatic DCT-branch selection.
AUTO_TUNE_ROUGHNESS_WEIGHT = 0.15  # Weight of curve roughness in the automatic-tuning score; larger values favor smoother curves.
REG_SCALE_FLOOR_QUANTILE = 10.0  # Lower quantile for the normalized error scale; prevents small high-frequency admittance values from receiving excessive weight.
# Plot-style settings.
RAW_LINE_WIDTH = 3.0  # Line width of the raw reference curve.
MODEL_LINE_WIDTH = 4.0  # Line width of the DCT/model prediction curve.
INPUT_POINT_SIZE = 80  # Input-point marker area used in DCT figures.
INPUT_POINT_EDGE_WIDTH = 1.8  # Edge width of input scatter markers.
NYQUIST_ERR_LINEWIDTH = 1.8  # Line width of Nyquist error bars.
NYQUIST_ERR_CAPSIZE = 3.0  # Capsize of Nyquist error bars.
CI_ALPHA = 0.20  # Transparency of the credible-interval shading.
TITLE_FONT_SIZE = 52  # Figure-title font size.
mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Microsoft YaHei", "SimHei"],
        "axes.unicode_minus": False,
        "font.size": 36,
        "axes.labelsize": 48,
        "xtick.labelsize": 36,
        "ytick.labelsize": 36,
        "legend.fontsize": 32,
        "axes.spines.right": False,
        "axes.spines.top": False,
        "axes.linewidth": 3,
        "legend.frameon": False,
    }
)
warnings.filterwarnings("ignore", category=RuntimeWarning)


### Cell 03 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: DCT-GPHT unbounded processing of experimental impedance data, dual-branch export, comparison of four condition types, and result saving.

- Function descriptions:
  - `ensure_dir(path)`: Takes a directory path, creates missing directories, and returns the same `Path` object for later table or figure output.
  - `stable_seed(*parts)`: Builds a reproducible random seed from sample names, condition labels, and other identifiers.
  - `safe_name(text, max_len)`: Takes a sample or condition name, removes invalid filename characters, limits its length, and returns a safe filename stem.
  - `compact_condition_name(condition)`: Converts an internal condition name into a shorter label suitable for figure filenames.
  - `figure_file_stem(sample, condition, used_stems)`: Builds a unique PNG filename stem from the sample name, DCT label, and degraded-condition label.
  - `safe_sheet_name(base, used)`: Takes a candidate sheet name and the set of used names, and returns a valid, unique Excel sheet name.

In [ ]:
@dataclass(frozen=True)
class Condition:
    name: str
    kind: str
    value: float | tuple[float, float] | None = None
def ensure_dir(path: Path) -> Path:
    """Create output directories.
    Inputs: path as need of directorypath.
    Outputs: one Path object.
    Purpose: Excel or PNG beforedirectory.
    """
    path.mkdir(parents=True, exist_ok=True)
    return path
def stable_seed(*parts: object) -> int:
    """Build a reproducible random seed from identifying strings.
    
    Inputs:
        parts: sample names, condition labels, or other identifiers.
    Outputs:
        32-bit integer random seed.
    Purpose:
        Keep noisy-condition generation reproducible for the same sample and condition.
    """
    text = "|".join(map(str, parts))
    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
    return (int(digest[:12], 16) + RANDOM_SEED) % (2**32 - 1)
def safe_name(text: object, max_len: int = 80) -> str:
    """Clean file names.
    Inputs: text as sample name or condition name, max_len as maximum retained length.
    Outputs: used for Windows file name of.
    Purpose: path and, to preventsavefigurewhenerrors.
    """
    s = str(text).strip()
    s = re.sub(r'[\\/:*?"<>|]+', "_", s)
    s = re.sub(r"\s+", "_", s)
    return s[:max_len] if s else "sample"
def compact_condition_name(condition: str) -> str:
    """Convert an internal condition name into a compact file-name token.
    
    Inputs:
        condition: internal condition label, such as raw, noisy_0p1, sparse_3, or limited_20pct.
    Outputs:
        Short cleaned string suitable for figure filenames.
    Purpose:
        Support consistent figure naming without changing the underlying condition labels.
    """
    return safe_name(str(condition).replace("_", ""))
def figure_file_stem(sample: str, condition: str, used_stems: set[str] | None = None) -> str:
    """Build a unique PNG filename stem for one DCT-GPHT figure.
    
    Inputs:
        sample: Excel sheet name.
        condition: degraded-condition label.
        used_stems: set of filename stems already used in the current run.
    Outputs:
        Unique filename stem following the sample + DCT + condition convention.
    Purpose:
        Prevent figure overwriting during batch processing.
    """
    base = f"{safe_name(sample)}DCT{compact_condition_name(condition)}"
    if used_stems is None:
        return base
    stem = base
    idx = 2
    while stem.lower() in used_stems:
        stem = f"{base}_{idx:02d}"
        idx += 1
    used_stems.add(stem.lower())
    return stem
def safe_sheet_name(base: str, used: set[str]) -> str:
    """Generate a valid and unique Excel sheet name.
    Inputs: base as candidate sheet name; used as of sheet nameset.
    Outputs: 31 repeatedly of sheet name.
    Purpose: Excel for sheet name and, thisfunctionunifiedprocessing.
    """
    cleaned = re.sub(r"[\[\]:*?/\\]", "_", str(base))[:31] or "sheet"
    name = cleaned
    idx = 1
    while name in used:
        suffix = f"_{idx}"
        name = cleaned[: 31 - len(suffix)] + suffix
        idx += 1
    used.add(name)
    return name


### Cell 04 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: DCT-GPHT unbounded processing of experimental impedance data, dual-branch export, comparison of four condition types, and result saving.

- Function descriptions:
  - `standardize_impedance_frame(df)`: Reads frequency, real impedance, and imaginary impedance from the first three columns and returns a standardized DataFrame with impedance and admittance columns.
  - `make_conditions()`: Generates the list of raw, noisy, sparse, and limited conditions from the global configuration.
  - `apply_condition(raw, condition, sample_name)`: Takes the raw impedance data and one degraded condition, and returns the corresponding raw/noisy/sparse/limited input spectrum.

In [ ]:
def standardize_impedance_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize one experimental impedance sheet.
    
    Inputs:
        df: DataFrame read from one Excel sheet; the first three columns are interpreted as frequency, Re(Z), and Im(Z).
    Outputs:
        DataFrame containing impedance columns (freq, re, imag, neg_imag) and admittance-domain columns (Y_re_mS, Y_imag_mS, neg_Y_imag_mS).
    Purpose:
        Clean invalid values, sort and deduplicate frequencies, and prepare both impedance-domain and admittance-domain data for DCT-GPHT.
    """
    if df.shape[1] < 3:
        raise ValueError("The input sheet must contain at least three columns, ordered as freq, re, and imag.")
    # The workbook may contain credible-interval columns; here only the first three columns are read as specified.
    freq = df.iloc[:, 0].to_numpy(dtype=float)
    z_re = df.iloc[:, 1].to_numpy(dtype=float)
    z_im = df.iloc[:, 2].to_numpy(dtype=float)
    out = pd.DataFrame(
        {
            "freq": freq,
            "re": z_re,
            "imag": z_im,
        }
    )
    valid = np.isfinite(out["freq"]) & np.isfinite(out["re"]) & np.isfinite(out["imag"]) & (out["freq"] > 0)
    out = out.loc[valid].copy()
    out = out.sort_values("freq").drop_duplicates("freq", keep="first").reset_index(drop=True)
    if len(out) < MIN_POINTS:
        raise ValueError(f"Insufficient valid impedance points: {len(out)}")
    out["neg_imag"] = -out["imag"]
    y = ADMITTANCE_SCALE / (out["re"].to_numpy(float) + 1j * out["imag"].to_numpy(float))
    out["Y_re_mS"] = y.real
    out["Y_imag_mS"] = y.imag
    out["neg_Y_imag_mS"] = -y.imag
    return out
def make_conditions() -> list[Condition]:
    """Generate the list of degraded conditions requested by the user configuration.
    
    Inputs:
        Uses INCLUDE_RAW_ANALYSIS, NOISE_LEVELS, SPARSE_RATIOS, LIMITED_PERCENT_LIST, and LIMITED_RANGE_LIST.
    Outputs:
        List of Condition objects.
    Purpose:
        Build raw, noisy, sparse, and limited inputs in a unified way before batch processing.
    """
    out: list[Condition] = []
    if INCLUDE_RAW_ANALYSIS:
        out.append(Condition("raw", "raw", None))
    for ratio in SPARSE_RATIOS:
        out.append(Condition(f"sparse_{int(ratio)}", "sparse", int(ratio)))
    for pct in LIMITED_PERCENT_LIST:
        out.append(Condition(f"limited_{int(round(float(pct) * 100))}pct", "limited_pct", float(pct)))
    for lo, hi in LIMITED_RANGE_LIST:
        out.append(Condition(f"limited_range_{lo:.0e}_{hi:.0e}", "limited_range", (float(lo), float(hi))))
    for level in NOISE_LEVELS:
        out.append(Condition(f"noisy_{level:g}".replace(".", "p"), "noisy", float(level)))
    return out
def apply_condition(raw: pd.DataFrame, condition: Condition, sample_name: str) -> pd.DataFrame:
    """Apply one degraded condition to the raw impedance spectrum.
    
    Inputs:
        raw or data_raw: raw impedance data for one sample.
        condition: Condition object generated by make_conditions.
        sample_name: sample/sheet name used to create reproducible noisy inputs when applicable.
    Outputs:
        DataFrame or dictionary with the same impedance fields as the input.
    Purpose:
        Copy raw data, add multiplicative relative noise, perform sparse sampling, or keep the requested frequency subset.
    """
    if condition.kind == "raw":
        return raw.copy()
    freq = raw["freq"].to_numpy(float)
    re_v = raw["re"].to_numpy(float)
    im_v = raw["imag"].to_numpy(float)
    if condition.kind == "noisy":
        level = float(condition.value)
        rng = np.random.default_rng(stable_seed(sample_name, condition.name))
        out = raw.copy()
        out["re"] = re_v * (1.0 + rng.normal(0.0, level, len(raw)))
        out["imag"] = im_v * (1.0 + rng.normal(0.0, level, len(raw)))
        return standardize_impedance_frame(out[["freq", "re", "imag"]])
    if condition.kind == "sparse":
        ratio = int(condition.value)
        idx = np.unique(np.r_[np.arange(0, len(raw), ratio), len(raw) - 1])
        return raw.iloc[idx].reset_index(drop=True)
    if condition.kind == "limited_pct":
        pct = float(condition.value)
        trim = int(math.floor(len(raw) * pct / 2.0))
        trim = min(trim, max(0, (len(raw) - MIN_POINTS) // 2))
        if trim <= 0:
            return raw.copy()
        return raw.iloc[trim : len(raw) - trim].reset_index(drop=True)
    if condition.kind == "limited_range":
        lo, hi = condition.value
        mask = (freq >= float(lo)) & (freq <= float(hi))
        out = raw.loc[mask].copy()
        if len(out) < MIN_POINTS:
            raise ValueError(f"{condition.name} retains only {len(out)} points, fewer than the minimum requirement.")
        return out.reset_index(drop=True)
    raise ValueError(f"Unsupported degraded-condition type: {condition.kind}")


### Cell 05 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: DCT-GPHT unbounded processing of experimental impedance data, dual-branch export, comparison of four condition types, and result saving.

- Function descriptions:
  - `cholesky_with_jitter(matrix)`: Adds diagonal jitter when needed and returns a stable lower-triangular Cholesky factor.
  - `kernel_matrix(omega_m, omega_n, opts, block)`: Calls the GP-HT unbounded kernel to generate the requested covariance block.
  - `sigma_floor_from_factor(scale, factor)`: Converts a relative noise-floor factor into the sigma_n lower bound on the current admittance scale.

In [ ]:
def cholesky_with_jitter(matrix: np.ndarray) -> np.ndarray:
    """Compute a stable Cholesky decomposition with adaptive jitter.
    
    Inputs:
        matrix: covariance matrix to decompose.
    Outputs:
        Lower-triangular Cholesky factor.
    Purpose:
        Improve numerical stability and fall back to nearest positive-definite correction when needed.
    """
    matrix = (matrix + matrix.T) / 2.0
    scale = max(float(np.mean(np.diag(matrix))), 1.0)
    for power in range(10):
        jitter = (10.0**power) * 1e-12 * scale
        try:
            return np.linalg.cholesky(matrix + jitter * np.eye(matrix.shape[0]))
        except np.linalg.LinAlgError:
            continue
    return np.linalg.cholesky(gpht_unbounded.nearest_PD(matrix))
def kernel_matrix(omega_m: np.ndarray, omega_n: np.ndarray, opts: dict, block: str) -> np.ndarray:
    """Generate a GP-HT unbounded/DCT covariance block.
    
    Inputs:
        omega_m, omega_n: row and column angular-frequency arrays.
        opts: kernel options including sigma_DCT, sigma_SB, ell, tau_min, DCT, SB, and SB_ker_type.
        block: covariance block name, such as re, im, re-im, or im-re.
    Outputs:
        Covariance matrix for the requested block.
    Purpose:
        Call GP_hilbert_unbounded.mat_K directly instead of reimplementing the DCT and Hilbert cross-covariance formulas.
    """
    return gpht_unbounded.mat_K(np.asarray(omega_m, dtype=float), np.asarray(omega_n, dtype=float), opts, block)
def sigma_floor_from_factor(scale: float, factor: float) -> float:
    """Convert a relative noise-floor factor into an absolute sigma_n floor.
    
    Inputs:
        scale: admittance-scale estimate for the current training data.
        factor: relative floor factor.
    Outputs:
        Lower bound for sigma_n in GP likelihood optimization.
    Purpose:
        Control overfitting by setting a minimum observation-noise level.
    """
    return max(scale * float(factor), scale * 1e-8, 1e-12)


### Cell 06 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: DCT-GPHT unbounded processing of experimental impedance data, dual-branch export, comparison of four condition types, and result saving.

- Function descriptions:
  - `initial_options(freq, y, sigma_n_floor_factor)`: Builds initial DCT-GPHT kernel options, log-theta values, and data scale from the training data.
  - `nmll_from_logtheta(logtheta, y, omega, opts_in, block)`: Computes the negative log marginal likelihood for the current log hyperparameters.
  - `optimize_branch(freq, y_train, block, sigma_n_floor_factor)`: Optimizes sigma_n, sigma_DCT, sigma_SB, and ell for one DCT-GPHT input branch.

In [ ]:
def initial_options(freq: np.ndarray, y: np.ndarray, sigma_n_floor_factor: float = 1e-8) -> tuple[dict, np.ndarray, float]:
    """Generate initial DCT-GPHT hyperparameters from the training data.
    
    Inputs:
        freq: training frequencies in Hz.
        y: training-side admittance values; Y_imag for imInput or centered Y_re for reInput.
        sigma_n_floor_factor: candidate noise-floor factor relative to the admittance scale.
    Outputs:
        opts: initial DCT/SB kernel-parameter dictionary.
        theta0: initial log(sigma_n, sigma_DCT, sigma_SB, ell).
        scale: data scale used for optimization bounds.
    Purpose:
        Initialize the kernel and optimizer consistently with the current frequency range and data scale.
    """
    scale = max(float(np.nanstd(y)), float(np.nanmean(np.abs(y))) * 1e-3, 1e-9)
    tau_min = float(TAU_MIN) if TAU_MIN is not None else 0.2 / float(np.max(freq))
    opts = {
        "sigma_DCT": max(scale * 0.8, 1e-9),
        "sigma_SB": max(scale * 0.3, 1e-9),
        "ell": 1.0,
        "tau_min": tau_min,
        "DCT": True,
        "SB": True,
        "SB_ker_type": "IQ",
    }
    sigma_n0 = max(float(np.nanmean(np.abs(y))) * 5e-4, scale * 1e-3, sigma_floor_from_factor(scale, sigma_n_floor_factor), 1e-9)
    theta0 = np.log([sigma_n0, opts["sigma_DCT"], opts["sigma_SB"], opts["ell"]])
    return opts, theta0, scale
def nmll_from_logtheta(logtheta: np.ndarray, y: np.ndarray, omega: np.ndarray, opts_in: dict, block: str) -> float:
    """Compute the negative log marginal likelihood for a given log-hyperparameter vector.
    
    Inputs:
        logtheta: log(sigma_n, sigma_DCT, sigma_SB, ell).
        y: training-side admittance values.
        omega: training angular frequencies.
        opts_in: fixed kernel options.
        block: training covariance block.
    Outputs:
        Scalar NMLL value; returns inf when the covariance matrix cannot be decomposed.
    Purpose:
        Provide the objective minimized during DCT-GPHT hyperparameter optimization.
    """
    sigma_n, sigma_dct, sigma_sb, ell = np.exp(logtheta)
    opts = dict(opts_in)
    opts.update({"sigma_DCT": sigma_dct, "sigma_SB": sigma_sb, "ell": ell})
    k = kernel_matrix(omega, omega, opts, block)
    k = k + (sigma_n**2) * np.eye(len(omega))
    try:
        lmat = cholesky_with_jitter(k)
        alpha = cho_solve((lmat, True), y, check_finite=False)
    except Exception:
        return np.inf
    return float(0.5 * y @ alpha + np.sum(np.log(np.diag(lmat))))
def optimize_branch(freq: np.ndarray, y_train: np.ndarray, block: str, sigma_n_floor_factor: float = 1e-8) -> tuple[dict, dict]:
    """Optimize one DCT-GPHT input branch.
    
    Inputs:
        freq: training frequencies in Hz.
        y_train: input-side admittance values for the current branch.
        block: training covariance block, im for imInput or re for reInput.
        sigma_n_floor_factor: candidate noise-floor factor.
    Outputs:
        opts: optimized kernel and noise parameters.
        log: optimizer status, NMLL, hyperparameters, and diagnostic message.
    Purpose:
        Learn sigma_n, sigma_DCT, sigma_SB, and ell for one candidate noise floor.
    """
    opts, theta0, scale = initial_options(freq, y_train, sigma_n_floor_factor)
    sigma_n_floor = sigma_floor_from_factor(scale, sigma_n_floor_factor)
    lower = np.log([sigma_n_floor, max(scale * 1e-5, 1e-12), max(scale * 1e-5, 1e-12), 1e-3])
    upper = np.log([max(scale * 2.0, 1e-9), max(scale * 1e3, 1e-9), max(scale * 1e3, 1e-9), 1e3])
    omega = 2.0 * math.pi * freq
    candidates = []
    res = minimize(
        nmll_from_logtheta,
        theta0,
        args=(y_train, omega, opts, block),
        method="L-BFGS-B",
        bounds=list(zip(lower, upper)),
        options={"maxiter": OPT_MAXITER, "ftol": 1e-8},
    )
    candidates.append(("L-BFGS-B", res))
    if (not res.success) or (not np.isfinite(res.fun)):
        res_powell = minimize(
            nmll_from_logtheta,
            np.clip(res.x if np.all(np.isfinite(res.x)) else theta0, lower, upper),
            args=(y_train, omega, opts, block),
            method="Powell",
            bounds=list(zip(lower, upper)),
            options={"maxiter": max(OPT_MAXITER, 160), "xtol": 1e-5, "ftol": 1e-7},
        )
        candidates.append(("Powell", res_powell))
    method, res = min(
        candidates,
        key=lambda item: item[1].fun if np.isfinite(item[1].fun) else np.inf,
    )
    sigma_n, sigma_dct, sigma_sb, ell = np.exp(res.x)
    opts.update({"sigma_DCT": sigma_dct, "sigma_SB": sigma_sb, "ell": ell})
    finite_result = bool(np.isfinite(res.fun))
    log = {
        "optimizer": method,
        "sigma_n": sigma_n,
        "sigma_DCT": sigma_dct,
        "sigma_SB": sigma_sb,
        "ell": ell,
        "tau_min": opts["tau_min"],
        "sigma_n_floor_factor": sigma_n_floor_factor,
        "sigma_n_floor": sigma_n_floor,
        "nmll": float(res.fun) if np.isfinite(res.fun) else np.nan,
        "success": finite_result,
        "optimization_success": bool(res.success),
        "message": f"{method}: {res.message}",
    }
    return opts | {"sigma_n": sigma_n}, log


### Cell 07 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: DCT-GPHT unbounded processing of experimental impedance data, dual-branch export, comparison of four condition types, and result saving.

- Function descriptions:
  - `posterior_component(omega_train, y_train, omega_pred, opts, train_block, target_block, cross_block, include_noise)`: Computes posterior mean and standard deviation for either same-side regression or cross-component prediction.
  - `local_admittance_scale(y_complex)`: Builds a pointwise normalization scale from the complex-admittance magnitude.
  - `normalized_regression_metrics(y_obs, y_fit, scale)`: Computes normalized NRMSE, R2, and maximum normalized residual for same-side input regression.
  - `normalized_curve_roughness(freq, y_fit, scale)`: Computes normalized second-difference roughness to detect excessive curve oscillation.
  - `auto_tuned_optimize_branch(freq, y_train, block, local_scale)`: Selects the smoothing/noise-floor setting that balances normalized regression error and curve roughness.
  - `z_stats_from_y(freq_pred, y_re_mu, y_re_sd, y_im_mu, y_im_sd, rng)`: Propagates admittance-domain posterior uncertainty into the impedance domain with Monte Carlo sampling.
  - `add_sigma_columns(df, re_col, im_col, sigma_re_col, sigma_im_col, prefix)`: Adds standard-deviation and credible-interval columns to exported impedance-domain results.

In [ ]:
def posterior_component(
    omega_train: np.ndarray,
    y_train: np.ndarray,
    omega_pred: np.ndarray,
    opts: dict,
    train_block: str,
    target_block: str,
    cross_block: str,
    include_noise: bool,
) -> tuple[np.ndarray, np.ndarray]:
    """Compute posterior mean and standard deviation for one GP-HT component.
    
    Inputs:
        omega_train: training angular frequencies.
        y_train: training-side values.
        omega_pred: prediction angular frequencies.
        opts: optimized kernel and noise parameters.
        train_block, target_block, cross_block: covariance-block names for regression or cross prediction.
        include_noise: whether to add observation noise to the prediction variance.
    Outputs:
        Posterior mean and standard deviation on the prediction grid.
    Purpose:
        Use the same conditional-Gaussian calculation for input-side regression and opposite-side Hilbert prediction.
    """
    k_train = kernel_matrix(omega_train, omega_train, opts, train_block)
    k_full = k_train + (opts["sigma_n"] ** 2) * np.eye(len(omega_train))
    lmat = cholesky_with_jitter(k_full)
    alpha = cho_solve((lmat, True), y_train, check_finite=False)
    k_cross = kernel_matrix(omega_train, omega_pred, opts, cross_block)
    mu = k_cross.T @ alpha
    v = solve_triangular(lmat, k_cross, lower=True, check_finite=False)
    k_pred_diag = np.diag(kernel_matrix(omega_pred, omega_pred, opts, target_block))
    var = k_pred_diag - np.sum(v * v, axis=0)
    if include_noise:
        var = var + opts["sigma_n"] ** 2
    sd = np.sqrt(np.maximum(var, 0.0))
    return mu, sd
def local_admittance_scale(y_complex: np.ndarray) -> np.ndarray:
    """Generate a pointwise normalization scale from complex admittance.
    
    Inputs:
        y_complex: complex admittance values converted from the current training impedance.
    Outputs:
        Positive scale array with the same length as y_complex.
    Purpose:
        Normalize errors without letting very small admittance values dominate residual metrics.
    """
    mag = np.abs(np.asarray(y_complex, dtype=complex))
    finite = mag[np.isfinite(mag)]
    if finite.size == 0:
        return np.ones_like(mag, dtype=float)
    floor = max(float(np.nanpercentile(finite, REG_SCALE_FLOOR_QUANTILE)), float(np.nanmedian(finite)) * 1e-3, 1e-12)
    return np.maximum(mag, floor)
def normalized_regression_metrics(y_obs: np.ndarray, y_fit: np.ndarray, scale: np.ndarray) -> dict:
    """Compute normalized regression metrics for the input side.
    
    Inputs:
        y_obs: observed training values for the current branch.
        y_fit: same-side GP regression mean on the training frequencies.
        scale: pointwise normalization scale from local_admittance_scale.
    Outputs:
        Dictionary containing normalized_rmse, normalized_r2, and max_abs_norm_residual.
    Purpose:
        Evaluate same-side regression quality on a scale that is comparable across frequency regions.
    """
    y_obs = np.asarray(y_obs, dtype=float)
    y_fit = np.asarray(y_fit, dtype=float)
    scale = np.asarray(scale, dtype=float)
    good = np.isfinite(y_obs) & np.isfinite(y_fit) & np.isfinite(scale) & (scale > 0)
    if np.count_nonzero(good) < 3:
        return {"normalized_rmse": np.nan, "normalized_r2": np.nan, "max_abs_norm_residual": np.nan}
    obs = y_obs[good]
    fit = y_fit[good]
    sc = scale[good]
    residual = (fit - obs) / sc
    weights = 1.0 / np.maximum(sc, 1e-12) ** 2
    weighted_mean = float(np.average(obs, weights=weights))
    centered = (obs - weighted_mean) / sc
    sse = float(np.sum(residual**2))
    sst = float(np.sum(centered**2))
    r2 = 1.0 - sse / max(sst, 1e-12)
    return {
        "normalized_rmse": float(np.sqrt(np.mean(residual**2))),
        "normalized_r2": float(r2),
        "max_abs_norm_residual": float(np.max(np.abs(residual))),
    }
def normalized_curve_roughness(freq: np.ndarray, y_fit: np.ndarray, scale: np.ndarray) -> float:
    """Compute normalized second-difference roughness of a regression curve.
    
    Inputs:
        freq: frequency grid.
        y_fit: fitted or predicted curve.
        scale: pointwise normalization scale.
    Outputs:
        Scalar roughness value.
    Purpose:
        Penalize excessive point-to-point oscillation during automatic smoothing selection.
    """
    freq = np.asarray(freq, dtype=float)
    y_fit = np.asarray(y_fit, dtype=float)
    scale = np.asarray(scale, dtype=float)
    good = np.isfinite(freq) & np.isfinite(y_fit) & np.isfinite(scale) & (freq > 0) & (scale > 0)
    if np.count_nonzero(good) < 5:
        return 0.0
    order = np.argsort(freq[good])
    norm_curve = y_fit[good][order] / scale[good][order]
    return float(np.sqrt(np.mean(np.diff(norm_curve, n=2) ** 2)))
def auto_tuned_optimize_branch(freq: np.ndarray, y_train: np.ndarray, block: str, local_scale: np.ndarray) -> tuple[dict, dict]:
    """Select a DCT-GPHT smoothing/noise setting using normalized regression quality.
    
    Inputs:
        freq: training frequencies.
        y_train: input-side admittance values.
        block: training covariance block.
        local_scale: pointwise normalization scale.
    Outputs:
        Optimized options and a log containing candidate scores and regression metrics.
    Purpose:
        Test a small set of sigma_n floor candidates and choose the one balancing NRMSE and roughness.
    """
    factors = SIGMA_N_FLOOR_FACTORS if AUTO_TUNE_SIGMA_N_FLOOR else [SIGMA_N_FLOOR_FACTORS[0]]
    omega = 2.0 * math.pi * freq
    best: tuple[float, dict, dict] | None = None
    candidate_rows = []
    for factor in factors:
        opts, log = optimize_branch(freq, y_train, block, sigma_n_floor_factor=float(factor))
        y_fit, _ = posterior_component(omega, y_train, omega, opts, block, block, block, include_noise=False)
        metrics = normalized_regression_metrics(y_train, y_fit, local_scale)
        roughness = normalized_curve_roughness(freq, y_fit, local_scale)
        score = metrics["normalized_rmse"] + AUTO_TUNE_ROUGHNESS_WEIGHT * roughness
        log.update(metrics)
        log.update({"normalized_roughness": roughness, "auto_tune_score": float(score)})
        candidate_rows.append(f"{factor:g}|rmse={metrics['normalized_rmse']:.4g}|rough={roughness:.4g}|score={score:.4g}")
        if best is None or score < best[0]:
            best = (float(score), opts, log)
    if best is None:
        raise RuntimeError("DCT-GPHT automatic tuning did not converge.")
    _, best_opts, best_log = best
    best_log["auto_tune_enabled"] = bool(AUTO_TUNE_SIGMA_N_FLOOR)
    best_log["auto_tune_candidates"] = "; ".join(candidate_rows)
    return best_opts, best_log
def z_stats_from_y(
    freq_pred: np.ndarray,
    y_re_mu: np.ndarray,
    y_re_sd: np.ndarray,
    y_im_mu: np.ndarray,
    y_im_sd: np.ndarray,
    rng: np.random.Generator,
) -> pd.DataFrame:
    """Convert admittance-domain posterior statistics into impedance-domain statistics.
    
    Inputs:
        freq_pred: prediction frequencies.
        y_re_mu/y_re_sd: posterior mean and standard deviation of admittance real part.
        y_im_mu/y_im_sd: posterior mean and standard deviation of admittance imaginary part.
        rng: random-number generator for Monte Carlo propagation.
    Outputs:
        DataFrame containing admittance statistics, impedance-domain means, standard deviations, and quantiles.
    Purpose:
        Propagate uncertainty through the nonlinear transformation from admittance to impedance.
    """
    y_mean = y_re_mu + 1j * y_im_mu
    z_from_mean_y = ADMITTANCE_SCALE / y_mean
    re_samples = rng.normal(y_re_mu[:, None], y_re_sd[:, None], size=(len(freq_pred), MC_SAMPLES))
    im_samples = rng.normal(y_im_mu[:, None], y_im_sd[:, None], size=(len(freq_pred), MC_SAMPLES))
    y_samples = re_samples + 1j * im_samples
    small = np.abs(y_samples) < 1e-12
    y_samples[small] = np.nan + 1j * np.nan
    z_samples = ADMITTANCE_SCALE / y_samples
    z_re = np.nanmean(z_samples.real, axis=1)
    z_im = np.nanmean(z_samples.imag, axis=1)
    z_re_sd = np.nanstd(z_samples.real, axis=1, ddof=1)
    z_im_sd = np.nanstd(z_samples.imag, axis=1, ddof=1)
    q = np.nanpercentile(z_samples.real, [2.5, 50.0, 97.5], axis=1)
    qi = np.nanpercentile(z_samples.imag, [2.5, 50.0, 97.5], axis=1)
    return pd.DataFrame(
        {
            "freq_pred": freq_pred,
            "Y_re_mean_mS": y_re_mu,
            "Y_imag_mean_mS": y_im_mu,
            "neg_Y_imag_mean_mS": -y_im_mu,
            "sigma_Y_re_mS": y_re_sd,
            "sigma_Y_imag_mS": y_im_sd,
            "re_from_meanY": z_from_mean_y.real,
            "imag_from_meanY": z_from_mean_y.imag,
            "neg_imag_from_meanY": -z_from_mean_y.imag,
            "re_mean": z_re,
            "imag_mean": z_im,
            "neg_imag_mean": -z_im,
            "sigma_re": z_re_sd,
            "sigma_imag": z_im_sd,
            "re_q025": q[0],
            "re_q50": q[1],
            "re_q975": q[2],
            "imag_q025": qi[0],
            "imag_q50": qi[1],
            "imag_q975": qi[2],
            "neg_imag_q025": -qi[2],
            "neg_imag_q50": -qi[1],
            "neg_imag_q975": -qi[0],
        }
    )
def add_sigma_columns(df: pd.DataFrame, re_col: str, im_col: str, sigma_re_col: str, sigma_im_col: str, prefix: str) -> pd.DataFrame:
    """Add sigma-based credible-interval columns to impedance-domain results.
    
    Inputs:
        df: result table from z_stats_from_y.
        re_col/im_col: impedance mean columns.
        sigma_re_col/sigma_im_col: corresponding standard-deviation columns.
        prefix: branch prefix, usually imInput or reInput.
    Outputs:
        Updated DataFrame with lower/upper interval columns.
    Purpose:
        Export both mean curves and uncertainty bands for later plotting.
    """
    out = df.copy()
    for k in (1, 2, 3):
        out[f"{prefix}_re_lower_{k}sigma"] = out[re_col] - k * out[sigma_re_col]
        out[f"{prefix}_re_upper_{k}sigma"] = out[re_col] + k * out[sigma_re_col]
        out[f"{prefix}_imag_lower_{k}sigma"] = out[im_col] - k * out[sigma_im_col]
        out[f"{prefix}_imag_upper_{k}sigma"] = out[im_col] + k * out[sigma_im_col]
        out[f"{prefix}_neg_imag_lower_{k}sigma"] = -(out[f"{prefix}_imag_upper_{k}sigma"])
        out[f"{prefix}_neg_imag_upper_{k}sigma"] = -(out[f"{prefix}_imag_lower_{k}sigma"])
    return out


### Cell 08 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: DCT-GPHT unbounded processing of experimental impedance data, dual-branch export, comparison of four condition types, and result saving.

- Function descriptions:
  - `run_dct_gpht_branch(sample, condition, observed, branch)`: Runs one DCT-GPHT branch for one sample and one degraded condition, returning posterior statistics and an optimization log.
  - `prefixed_branch_frame(branch, stats, log)`: Adds branch-specific prefixes to imInput or reInput result columns for side-by-side Excel export.

In [ ]:
def run_dct_gpht_branch(sample: str, condition: str, observed: pd.DataFrame, branch: str) -> tuple[pd.DataFrame, dict]:
    """Run one DCT-GPHT branch for one sample and one condition.
    
    Inputs:
        sample: Excel sheet name.
        condition: current degraded-condition label.
        observed: impedance data used for GP-HT training.
        branch: imInput or reInput.
    Outputs:
        Posterior statistics table and optimization log.
    Purpose:
        Run the admittance-domain DCT-GPHT calculation and convert results back to impedance-domain statistics.
    """
    freq_obs = observed["freq"].to_numpy(float)
    z_obs = observed["re"].to_numpy(float) + 1j * observed["imag"].to_numpy(float)
    y_obs = ADMITTANCE_SCALE / z_obs
    y_re = y_obs.real
    y_im = y_obs.imag
    fit_scale = local_admittance_scale(y_obs)
    omega_obs = 2.0 * math.pi * freq_obs
    freq_pred = np.logspace(np.log10(PRED_FREQ_MIN), np.log10(PRED_FREQ_MAX), N_PRED)
    omega_pred = 2.0 * math.pi * freq_pred
    if branch == "imInput":
        y_train = y_im
        opts, log = auto_tuned_optimize_branch(freq_obs, y_train, "im", fit_scale)
        y_im_mu, y_im_sd = posterior_component(omega_obs, y_train, omega_pred, opts, "im", "im", "im", include_noise=False)
        y_re_mu, y_re_sd = posterior_component(omega_obs, y_train, omega_pred, opts, "im", "re", "im-re", include_noise=True)
        y_re_obs_mu, _ = posterior_component(omega_obs, y_train, omega_obs, opts, "im", "re", "im-re", include_noise=True)
        target_offset = float(np.mean(y_re - y_re_obs_mu))
        y_re_mu = y_re_mu + target_offset
        output_kind = {"re_label": "re_pred", "imag_label": "imag_reg", "target_offset": target_offset, "train_offset": 0.0}
    elif branch == "reInput":
        train_offset = float(np.median(y_re))
        y_train = y_re - train_offset
        opts, log = auto_tuned_optimize_branch(freq_obs, y_train, "re", fit_scale)
        y_re_mu, y_re_sd = posterior_component(omega_obs, y_train, omega_pred, opts, "re", "re", "re", include_noise=False)
        y_re_mu = y_re_mu + train_offset
        y_im_mu, y_im_sd = posterior_component(omega_obs, y_train, omega_pred, opts, "re", "im", "re-im", include_noise=True)
        if APPLY_REINPUT_IMAG_OFFSET:
            y_im_obs_mu, _ = posterior_component(omega_obs, y_train, omega_obs, opts, "re", "im", "re-im", include_noise=True)
            target_offset = float(np.mean(y_im - y_im_obs_mu))
            y_im_mu = y_im_mu + target_offset
        else:
            target_offset = 0.0
        output_kind = {"re_label": "re_reg", "imag_label": "imag_pred", "target_offset": target_offset, "train_offset": train_offset}
    else:
        raise ValueError(f"Unsupported branch type: {branch}")
    rng = np.random.default_rng(stable_seed(sample, condition, branch, "posterior"))
    stats = z_stats_from_y(freq_pred, y_re_mu, y_re_sd, y_im_mu, y_im_sd, rng)
    stats = add_sigma_columns(stats, "re_mean", "imag_mean", "sigma_re", "sigma_imag", branch)
    log.update(output_kind)
    log.update({"sample": sample, "condition": condition, "branch": branch, "n_obs": len(observed), "f_min": np.min(freq_obs), "f_max": np.max(freq_obs)})
    return stats, log
def prefixed_branch_frame(branch: str, stats: pd.DataFrame, log: dict) -> pd.DataFrame:
    """Format one branch result for side-by-side Excel export.
    
    Inputs:
        branch: imInput or reInput label.
        stats: posterior statistics returned by run_dct_gpht_branch.
        log: branch-level optimization log.
    Outputs:
        DataFrame with branch-prefixed result columns.
    Purpose:
        Keep imInput and reInput outputs unambiguous when written into the same worksheet.
    """
    re_label = log["re_label"]
    im_label = log["imag_label"]
    out = pd.DataFrame({"freq_pred": stats["freq_pred"]})
    out[f"{branch}_Y_re_{re_label}_mS"] = stats["Y_re_mean_mS"]
    out[f"{branch}_Y_imag_{im_label}_mS"] = stats["Y_imag_mean_mS"]
    out[f"{branch}_neg_Y_imag_{im_label}_mS"] = stats["neg_Y_imag_mean_mS"]
    out[f"{branch}_sigma_Y_re_{re_label}_mS"] = stats["sigma_Y_re_mS"]
    out[f"{branch}_sigma_Y_imag_{im_label}_mS"] = stats["sigma_Y_imag_mS"]
    out[f"{branch}_{re_label}"] = stats["re_mean"]
    out[f"{branch}_{im_label}"] = stats["imag_mean"]
    out[f"{branch}_neg_{im_label}"] = stats["neg_imag_mean"]
    out[f"{branch}_sigma_{re_label}"] = stats["sigma_re"]
    out[f"{branch}_sigma_{im_label}"] = stats["sigma_imag"]
    out[f"{branch}_re_from_meanY"] = stats["re_from_meanY"]
    out[f"{branch}_imag_from_meanY"] = stats["imag_from_meanY"]
    out[f"{branch}_neg_imag_from_meanY"] = stats["neg_imag_from_meanY"]
    out[f"{branch}_re_q025"] = stats["re_q025"]
    out[f"{branch}_re_q50"] = stats["re_q50"]
    out[f"{branch}_re_q975"] = stats["re_q975"]
    out[f"{branch}_imag_q025"] = stats["imag_q025"]
    out[f"{branch}_imag_q50"] = stats["imag_q50"]
    out[f"{branch}_imag_q975"] = stats["imag_q975"]
    out[f"{branch}_neg_imag_q025"] = stats["neg_imag_q025"]
    out[f"{branch}_neg_imag_q50"] = stats["neg_imag_q50"]
    out[f"{branch}_neg_imag_q975"] = stats["neg_imag_q975"]
    for k in (1, 2, 3):
        out[f"{branch}_{re_label}_lower_{k}sigma"] = stats[f"{branch}_re_lower_{k}sigma"]
        out[f"{branch}_{re_label}_upper_{k}sigma"] = stats[f"{branch}_re_upper_{k}sigma"]
        out[f"{branch}_{im_label}_lower_{k}sigma"] = stats[f"{branch}_imag_lower_{k}sigma"]
        out[f"{branch}_{im_label}_upper_{k}sigma"] = stats[f"{branch}_imag_upper_{k}sigma"]
        out[f"{branch}_neg_{im_label}_lower_{k}sigma"] = stats[f"{branch}_neg_imag_lower_{k}sigma"]
        out[f"{branch}_neg_{im_label}_upper_{k}sigma"] = stats[f"{branch}_neg_imag_upper_{k}sigma"]
    # retainplottingcall of column name.
    if branch == "reInput":
        out["reInput_sigma_re_pred"] = out["reInput_sigma_re_reg"]
        out["reInput_sigma_imag_reg"] = out["reInput_sigma_imag_pred"]
    if branch == "imInput":
        out["imInput_sigma_re_reg"] = out["imInput_sigma_re_pred"]
        out["imInput_sigma_imag_pred"] = out["imInput_sigma_imag_reg"]
    return out


### Cell 09 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: DCT-GPHT unbounded processing of experimental impedance data, dual-branch export, comparison of four condition types, and result saving.

- Function descriptions:
  - `pad(values, n)`: of Series or DataFrame as specified, to facilitatesave.
  - `assemble_sheet(raw, observed, branch_tables, condition)`: raw, degraded input and of DCT-GPHT merge as one Excel sheet.

In [ ]:
def pad(values, n: int) -> np.ndarray:
    """tospecified.
    Inputs: values as onecolumndata; n as target.
    Outputs: as n of, column NaN.
    Purpose: rawdata, degradeddata and prediction, toone sheet
        whenneed.
    """
    arr = np.asarray(values)
    if len(arr) >= n:
        return arr[:n]
    if arr.dtype.kind in {"U", "S", "O"}:
        out = np.empty(n, dtype=object)
        out[:] = np.nan
        out[: len(arr)] = arr
        return out
    return np.pad(arr.astype(float), (0, n - len(arr)), constant_values=np.nan)
def assemble_sheet(raw: pd.DataFrame, observed: pd.DataFrame, branch_tables: dict[str, pd.DataFrame], condition: str) -> pd.DataFrame:
    """Assemble one Excel result sheet.
    Inputs:
        raw: raw impedance data.
        observed: impedance data under the current degraded condition.
        branch_tables: prefixed_branch_frame generate of one or table.
        condition: currentdegradedcondition name, degradeddatacolumn namebefore.
    Outputs:
        directlywrite Excel of table DataFrame.
    Purpose:
        one sheet inalsosave raw, currentdegradedcondition, freq_pred, imInput
        and / or reInput of laterto of datacolumn.
    """
    frames = []
    n = max([len(raw), len(observed)] + [len(v) for v in branch_tables.values()])
    raw_part = pd.DataFrame(
        {
            "raw_freq": pad(raw["freq"], n),
            "raw_re": pad(raw["re"], n),
            "raw_imag": pad(raw["imag"], n),
            "raw_neg_imag": pad(raw["neg_imag"], n),
            "raw_Y_re_mS": pad(raw["Y_re_mS"], n),
            "raw_Y_imag_mS": pad(raw["Y_imag_mS"], n),
            "raw_neg_Y_imag_mS": pad(raw["neg_Y_imag_mS"], n),
        }
    )
    cond_part = pd.DataFrame(
        {
            f"{condition}_freq": pad(observed["freq"], n),
            f"{condition}_re": pad(observed["re"], n),
            f"{condition}_imag": pad(observed["imag"], n),
            f"{condition}_neg_imag": pad(observed["neg_imag"], n),
            f"{condition}_Y_re_mS": pad(observed["Y_re_mS"], n),
            f"{condition}_Y_imag_mS": pad(observed["Y_imag_mS"], n),
            f"{condition}_neg_Y_imag_mS": pad(observed["neg_Y_imag_mS"], n),
        }
    )
    frames.extend([raw_part, cond_part])
    if branch_tables:
        first_table = next(iter(branch_tables.values()))
        if "freq_pred" in first_table:
            frames.append(pd.DataFrame({"freq_pred": pad(first_table["freq_pred"], n)}))
    for branch, table in branch_tables.items():
        frames.append(pd.DataFrame({col: pad(table[col], n) for col in table.columns if col != "freq_pred"}))
    return pd.concat(frames, axis=1)


### Cell 10 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: DCT-GPHT unbounded processing of experimental impedance data, dual-branch export, comparison of four condition types, and result saving.

- Function descriptions:
  - `plot_condition(sample, condition, raw, observed, branch_stats, used_figure_stems)`: Plots DCT-GPHT Nyquist, real-part Bode, and imaginary-part Bode panels for one sample and one condition.
  - `condition_label(condition)`: Takes a condition name and returns a short label suitable for titles or legends.
  - `plot_sample_grid(sample, raw, rows, used_figure_stems)`: Combines raw, noisy, sparse, and limited DCT-GPHT results for one sample into a multi-row by three-column figure.

In [ ]:
def plot_condition(
    sample: str,
    condition: str,
    raw: pd.DataFrame,
    observed: pd.DataFrame,
    branch_stats: dict[str, pd.DataFrame],
    used_figure_stems: set[str] | None = None,
) -> Path:
    """Plot and save DCT-GPHT results for one sample and one condition.
    
    Inputs:
        sample: sample/sheet name.
        condition: degraded-condition label.
        raw: raw impedance data.
        observed: degraded input data.
        branch_stats: posterior results for the selected branches.
        used_figure_stems: filename stems already used in this run.
    Outputs:
        Path of the saved PNG figure.
    Purpose:
        Draw Nyquist, real-part Bode, and -imaginary-part Bode panels with selected branches and credible intervals.
    """
    fig, axes = plt.subplots(1, 3, figsize=(38, 13), constrained_layout=True)
    fig.suptitle(f"{sample} {condition}", fontsize=TITLE_FONT_SIZE, fontweight="bold")
    colors = {"imInput": "#2f6fbb", "reInput": "#2f8f5b"}
    ax = axes[0]
    ax.plot(raw["re"], raw["neg_imag"], "-", color="0.25", lw=RAW_LINE_WIDTH, label="raw")
    ax.scatter(
        observed["re"],
        observed["neg_imag"],
        s=INPUT_POINT_SIZE,
        facecolor="white",
        edgecolor="#b03a2e",
        linewidth=INPUT_POINT_EDGE_WIDTH,
        label=condition,
    )
    for branch, stats in branch_stats.items():
        color = colors.get(branch, "tab:blue")
        ax.plot(stats["re_mean"], stats["neg_imag_mean"], "-", lw=MODEL_LINE_WIDTH, color=color, label=branch)
        idx = np.arange(0, len(stats), NYQUIST_ERR_STRIDE)
        ax.errorbar(
            stats["re_mean"].iloc[idx],
            stats["neg_imag_mean"].iloc[idx],
            xerr=CI_K_FOR_PLOT * stats["sigma_re"].iloc[idx],
            yerr=CI_K_FOR_PLOT * stats["sigma_imag"].iloc[idx],
            fmt="none",
            ecolor=color,
            elinewidth=NYQUIST_ERR_LINEWIDTH,
            alpha=0.35,
            capsize=NYQUIST_ERR_CAPSIZE,
        )
    ax.set_xlabel("Z' (ohm)")
    ax.set_ylabel("-Z'' (ohm)")
    ax.set_aspect("equal", adjustable="datalim")
    ax.legend(loc="best")
    ax = axes[1]
    ax.semilogx(raw["freq"], raw["re"], "-", color="0.25", lw=RAW_LINE_WIDTH, label="raw")
    ax.scatter(observed["freq"], observed["re"], s=INPUT_POINT_SIZE, facecolor="white", edgecolor="#b03a2e", linewidth=INPUT_POINT_EDGE_WIDTH, label=condition)
    for branch, stats in branch_stats.items():
        color = colors.get(branch, "tab:blue")
        ax.semilogx(stats["freq_pred"], stats["re_mean"], "-", lw=MODEL_LINE_WIDTH, color=color, label=branch)
        ax.fill_between(
            stats["freq_pred"],
            stats["re_mean"] - CI_K_FOR_PLOT * stats["sigma_re"],
            stats["re_mean"] + CI_K_FOR_PLOT * stats["sigma_re"],
            color=color,
            alpha=CI_ALPHA,
            linewidth=0,
        )
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Z' (ohm)")
    ax = axes[2]
    ax.semilogx(raw["freq"], raw["neg_imag"], "-", color="0.25", lw=RAW_LINE_WIDTH, label="raw")
    ax.scatter(observed["freq"], observed["neg_imag"], s=INPUT_POINT_SIZE, facecolor="white", edgecolor="#b03a2e", linewidth=INPUT_POINT_EDGE_WIDTH, label=condition)
    for branch, stats in branch_stats.items():
        color = colors.get(branch, "tab:blue")
        neg = stats["neg_imag_mean"]
        sig = stats["sigma_imag"]
        ax.semilogx(stats["freq_pred"], neg, "-", lw=MODEL_LINE_WIDTH, color=color, label=branch)
        ax.fill_between(stats["freq_pred"], neg - CI_K_FOR_PLOT * sig, neg + CI_K_FOR_PLOT * sig, color=color, alpha=CI_ALPHA, linewidth=0)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("-Z'' (ohm)")
    for ax in axes:
        ax.tick_params(width=3, length=10)
        for spine in ax.spines.values():
            spine.set_linewidth(3)
    png_path = FIG_DIR / f"{figure_file_stem(sample, condition, used_figure_stems)}.png"
    fig.savefig(png_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    return png_path
def condition_label(condition: str) -> str:
    """Generate English condition labels for the 4x3 summary figure.
    Inputs: condition as insidecondition name, raw, sparse_3, limited_20pct, noisy_0p1.
    Outputs: figurein of.
    Purpose: DCT/DRT figureand DRT 4x3 figurekeep.
    """
    if condition == "raw":
        return "Raw Data"
    if condition.startswith("sparse_"):
        return f"Sparse (1/{condition.split('_')[-1]})"
    if condition.startswith("limited_") and condition.endswith("pct"):
        pct = condition.split("_")[-1].replace("pct", "")
        return f"Limited ({pct}%)"
    if condition.startswith("limited_range_"):
        return "Limited Freq"
    if condition.startswith("noisy_"):
        level = condition.split("_", 1)[1].replace("p", ".")
        try:
            return f"Noisy ({float(level) * 100:g}%)"
        except ValueError:
            return "Noisy"
    return condition
def plot_sample_grid(
    sample: str,
    raw: pd.DataFrame,
    rows: list[dict],
    used_figure_stems: set[str] | None = None,
) -> Path:
    """Plot all DCT-GPHT conditions for one sample in a multi-row by three-column figure.
    
    Inputs:
        sample: sample/sheet name.
        raw: raw impedance data.
        rows: list of condition-specific observed data and branch results.
        used_figure_stems: filename stems already used in this run.
    Outputs:
        Path of the saved PNG summary figure.
    Purpose:
        Use one consistent 4 x 3 style figure for raw, noisy, sparse, and limited conditions.
    """
    if not rows:
        raise ValueError("No condition results are available for plotting.")
    n_rows = len(rows)
    fig, axes = plt.subplots(n_rows, 3, figsize=(34, max(9, 8 * n_rows)), constrained_layout=True)
    if n_rows == 1:
        axes = np.array([axes])
    colors = {"imInput": "#0072B2", "reInput": "#D55E00"}
    labels = {"imInput": "im-input", "reInput": "re-input"}
    raw_freq = raw["freq"].to_numpy(float)
    raw_re = raw["re"].to_numpy(float)
    raw_neg_im = raw["neg_imag"].to_numpy(float)
    for row_idx, item in enumerate(rows):
        condition = item["condition"]
        observed = item["observed"]
        branch_stats = item["branch_stats"]
        label = condition_label(condition)
        obs_freq = observed["freq"].to_numpy(float)
        obs_re = observed["re"].to_numpy(float)
        obs_neg_im = observed["neg_imag"].to_numpy(float)
        ax = axes[row_idx, 0]
        ax.plot(raw_re, raw_neg_im, color="black", alpha=0.18, lw=RAW_LINE_WIDTH, label="Raw reference")
        ax.plot(obs_re, obs_neg_im, "o", color="#b03a2e", markersize=max(6, INPUT_POINT_SIZE / 10), alpha=0.75, label="Input data")
        for branch, stats in branch_stats.items():
            color = colors.get(branch, "tab:blue")
            ax.plot(stats["re_mean"], stats["neg_imag_mean"], "-", lw=MODEL_LINE_WIDTH, color=color, label=labels.get(branch, branch))
            idx = np.arange(0, len(stats), max(1, NYQUIST_ERR_STRIDE))
            ax.errorbar(
                stats["re_mean"].iloc[idx],
                stats["neg_imag_mean"].iloc[idx],
                xerr=CI_K_FOR_PLOT * stats["sigma_re"].iloc[idx],
                yerr=CI_K_FOR_PLOT * stats["sigma_imag"].iloc[idx],
                fmt="none",
                ecolor=color,
                elinewidth=NYQUIST_ERR_LINEWIDTH,
                capsize=NYQUIST_ERR_CAPSIZE,
                alpha=0.35,
            )
        ax.set_ylabel(f"{label}\n-Z'' (Ohm)", fontfamily="Arial", fontsize=48, fontweight="bold")
        if row_idx == 0:
            ax.set_title("Nyquist Plot", fontfamily="Arial", fontsize=48)
        ax.legend(loc="best")
        ax.axis("equal")
        ax = axes[row_idx, 1]
        ax.semilogx(raw_freq, raw_re, color="black", alpha=0.18, lw=RAW_LINE_WIDTH)
        ax.semilogx(obs_freq, obs_re, "o", color="#b03a2e", markersize=max(6, INPUT_POINT_SIZE / 10), alpha=0.75)
        for branch, stats in branch_stats.items():
            color = colors.get(branch, "tab:blue")
            ax.semilogx(stats["freq_pred"], stats["re_mean"], "-", lw=MODEL_LINE_WIDTH, color=color, label=labels.get(branch, branch))
            ax.fill_between(
                stats["freq_pred"],
                stats["re_mean"] - CI_K_FOR_PLOT * stats["sigma_re"],
                stats["re_mean"] + CI_K_FOR_PLOT * stats["sigma_re"],
                color=color,
                alpha=CI_ALPHA,
                linewidth=0,
            )
        ax.set_ylabel("Z' (Ohm)", fontfamily="Arial", fontsize=48)
        if row_idx == 0:
            ax.set_title("Real Part Bode", fontfamily="Arial", fontsize=48)
            ax.legend(loc="best")
        ax = axes[row_idx, 2]
        ax.semilogx(raw_freq, raw_neg_im, color="black", alpha=0.18, lw=RAW_LINE_WIDTH)
        ax.semilogx(obs_freq, obs_neg_im, "o", color="#b03a2e", markersize=max(6, INPUT_POINT_SIZE / 10), alpha=0.75)
        for branch, stats in branch_stats.items():
            color = colors.get(branch, "tab:blue")
            neg = stats["neg_imag_mean"]
            sig = stats["sigma_imag"]
            ax.semilogx(stats["freq_pred"], neg, "-", lw=MODEL_LINE_WIDTH, color=color, label=labels.get(branch, branch))
            ax.fill_between(stats["freq_pred"], neg - CI_K_FOR_PLOT * sig, neg + CI_K_FOR_PLOT * sig, color=color, alpha=CI_ALPHA, linewidth=0)
        ax.set_ylabel("-Z'' (Ohm)", fontfamily="Arial", fontsize=48)
        if row_idx == 0:
            ax.set_title("Imaginary Part Bode", fontfamily="Arial", fontsize=48)
        if row_idx == n_rows - 1:
            axes[row_idx, 1].set_xlabel("Frequency (Hz)", fontfamily="Arial", fontsize=48)
            axes[row_idx, 2].set_xlabel("Frequency (Hz)", fontfamily="Arial", fontsize=48)
    for ax in axes.ravel():
        ax.tick_params(axis="both", labelsize=36, width=3, length=9)
        for tick_label in ax.get_xticklabels() + ax.get_yticklabels():
            tick_label.set_fontfamily("Arial")
        for spine in ax.spines.values():
            spine.set_linewidth(3)
    base = f"{safe_name(sample)}DCT4x3"
    stem = base
    if used_figure_stems is not None:
        idx = 2
        while stem.lower() in used_figure_stems:
            stem = f"{base}_{idx:02d}"
            idx += 1
        used_figure_stems.add(stem.lower())
    png_path = FIG_DIR / f"{stem}.png"
    fig.savefig(png_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    return png_path


### Cell 11 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: DCT-GPHT unbounded processing of experimental impedance data, dual-branch export, comparison of four condition types, and result saving.

- Function descriptions: 
  - `select_sheets(xls)`: Reads sheet names from the Excel workbook and filters samples by user-defined keywords and maximum count.


In [ ]:
def select_sheets(xls: pd.ExcelFile) -> list[str]:
    """Select worksheet names according to optional name filters and maximum-count settings.
    
    Inputs:
        xls_file or xls: an opened Excel workbook.
    Outputs:
        List of sheet names to analyze.
    Purpose:
        Read all sample sheets by default, or restrict the run for debugging or targeted analysis.
    """
    sheets = xls.sheet_names
    if SHEET_NAME_PATTERNS:
        selected = [s for s in sheets if any(pat in s for pat in SHEET_NAME_PATTERNS)]
    else:
        selected = list(sheets)
    if MAX_SHEETS is not None:
        selected = selected[:MAX_SHEETS]
    if not selected:
        raise ValueError(f"No sheet matched the filter keywords: {SHEET_NAME_PATTERNS}")
    return selected
# readworkbook.This cell does not write result files.
conditions = make_conditions()
xls = pd.ExcelFile(INPUT_EXCEL)
sheets = select_sheets(xls)
print("Total workbook sheets:", len(xls.sheet_names))
print("Number of selected sheets:", len(sheets))
print("Selected sheets:", sheets)
print("Degraded conditions:", [condition.name for condition in conditions])
print("Branches to run:", RUN_BRANCHES)
preview_raw = None
preview_sheet = None
for sheet_name in sheets:
    try:
        preview_raw = standardize_impedance_frame(pd.read_excel(xls, sheet_name=sheet_name))
        preview_sheet = sheet_name
        break
    except Exception as exc:
        print(f"Preview skipped: {sheet_name} | {exc}")
if preview_raw is None:
    raise ValueError("None of the selected sheets can be parsed as impedance data from the first three columns.")
print("First valid impedance sheet:", preview_sheet, "data shape:", preview_raw.shape)
display(preview_raw.head())


### Cell 12 - Experimental Parameter Configuration

This cell configures the input file, output directory, noise levels, sparse-sampling ratios, frequency-limitation/truncation settings, prediction-frequency range, and random seed.

- Purpose: DCT-GPHT unbounded processing of experimental impedance data, dual-branch export, comparison of four condition types, and result saving.
- Main parameters: RUN_SINGLE_CASE_DEBUG, DEBUG_SAMPLE, DEBUG_CONDITION_NAME.These variables control input/output paths, experimental conditions, frequency ranges, or plotting behavior.

In [ ]:
# Options: formalbatchbefore, debuggingonesample and onedegradedcondition.
# enable, RUN_SINGLE_CASE_DEBUG as True.
RUN_SINGLE_CASE_DEBUG = False
DEBUG_SAMPLE = sheets[0]
DEBUG_CONDITION_NAME = "raw"
if RUN_SINGLE_CASE_DEBUG:
    ensure_dir(FIG_DIR)
    raw_debug = standardize_impedance_frame(pd.read_excel(xls, sheet_name=DEBUG_SAMPLE))
    condition_map = {condition.name: condition for condition in conditions}
    observed_debug = apply_condition(raw_debug, condition_map[DEBUG_CONDITION_NAME], DEBUG_SAMPLE)
    branch_stats_debug = {}
    branch_tables_debug = {}
    log_debug = []
    for branch in RUN_BRANCHES:
        stats, log = run_dct_gpht_branch(DEBUG_SAMPLE, DEBUG_CONDITION_NAME, observed_debug, branch)
        branch_stats_debug[branch] = stats
        branch_tables_debug[branch] = prefixed_branch_frame(branch, stats, log)
        log_debug.append(log)
    debug_sheet = assemble_sheet(raw_debug, observed_debug, branch_tables_debug, DEBUG_CONDITION_NAME)
    debug_fig = plot_condition(DEBUG_SAMPLE, DEBUG_CONDITION_NAME, raw_debug, observed_debug, branch_stats_debug, set())
    display(debug_sheet.head())
    display(pd.DataFrame(log_debug))
    print("Debug figure saved to:", debug_fig)
else:
    print("Single-sample debugging was skipped. Set RUN_SINGLE_CASE_DEBUG to True to enable it.")


### Cell 13 - Function Definitions

This cell defines reusable functions for model generation, data degradation, GP-HT regression, parameter fitting, result export, or plotting.

- Purpose: DCT-GPHT unbounded processing of experimental impedance data, dual-branch export, comparison of four condition types, and result saving.

- Function descriptions:
  - `run_all()`: Reads all selected samples, runs all degraded conditions and selected branches, and exports Excel files, figures, and logs.

In [ ]:
def run_all() -> tuple[Path, Path, pd.DataFrame, pd.DataFrame]:
    """Run the full DCT-GPHT batch workflow.
    
    Inputs:
        Uses INPUT_EXCEL, selected sheets, condition settings, and branch settings.
    Outputs:
        Excel workbook, optimization log, figure index, and PNG figures.
    Purpose:
        Process all selected samples and degraded conditions with the requested DCT-GPHT branches.
    """
    ensure_dir(DATA_DIR)
    ensure_dir(FIG_DIR)
    conditions = make_conditions()
    if not conditions:
        raise ValueError("No degraded condition is enabled. Enable INCLUDE_RAW_ANALYSIS or define at least one condition.")
    if not RUN_BRANCHES:
        raise ValueError("RUN_BRANCHES is empty. Use ['imInput'], ['reInput'], or both.")
    xls = pd.ExcelFile(INPUT_EXCEL)
    sheets = select_sheets(xls)
    print(f"Total workbook sheets: {len(xls.sheet_names)}; selected for DCT-GPHT: {len(sheets)}")
    print("Selected sheets:", sheets)
    output_xlsx = DATA_DIR / f"{DATASET_NAME}_DCT_GPHT_unbounded_experimental_results.xlsx"
    used_sheet_names: set[str] = set()
    used_figure_stems: set[str] = set()
    log_rows: list[dict] = []
    fig_rows: list[dict] = []
    with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
        pd.DataFrame(
            {
                "parameter": [
                    "INPUT_EXCEL",
                    "RUN_ID",
                    "DATA_DIR",
                    "FIG_DIR",
                    "SHEET_NAME_PATTERNS",
                    "SKIP_INVALID_SHEETS",
                    "RUN_BRANCHES",
                    "NOISE_LEVELS",
                    "SPARSE_RATIOS",
                    "LIMITED_PERCENT_LIST",
                    "LIMITED_RANGE_LIST",
                    "PRED_FREQ_MIN",
                    "PRED_FREQ_MAX",
                    "N_PRED",
                    "ADMITTANCE_SCALE",
                    "TAU_MIN",
                    "MC_SAMPLES",
                    "CI_K_FOR_PLOT",
                    "FIG_DPI",
                    "APPLY_REINPUT_IMAG_OFFSET",
                    "AUTO_TUNE_SIGMA_N_FLOOR",
                    "SIGMA_N_FLOOR_FACTORS",
                    "AUTO_TUNE_ROUGHNESS_WEIGHT",
                    "REG_SCALE_FLOOR_QUANTILE",
                    "RAW_LINE_WIDTH",
                    "MODEL_LINE_WIDTH",
                    "INPUT_POINT_SIZE",
                    "INPUT_POINT_EDGE_WIDTH",
                    "NYQUIST_ERR_LINEWIDTH",
                    "NYQUIST_ERR_CAPSIZE",
                    "CI_ALPHA",
                    "TITLE_FONT_SIZE",
                ],
                "value": [
                    str(INPUT_EXCEL),
                    RUN_ID,
                    str(DATA_DIR),
                    str(FIG_DIR),
                    ", ".join(SHEET_NAME_PATTERNS) if SHEET_NAME_PATTERNS else "ALL",
                    SKIP_INVALID_SHEETS,
                    ", ".join(RUN_BRANCHES),
                    str(NOISE_LEVELS),
                    str(SPARSE_RATIOS),
                    str(LIMITED_PERCENT_LIST),
                    str(LIMITED_RANGE_LIST),
                    PRED_FREQ_MIN,
                    PRED_FREQ_MAX,
                    N_PRED,
                    ADMITTANCE_SCALE,
                    "auto: 0.2 / max(freq_obs)" if TAU_MIN is None else TAU_MIN,
                    MC_SAMPLES,
                    CI_K_FOR_PLOT,
                    FIG_DPI,
                    APPLY_REINPUT_IMAG_OFFSET,
                    AUTO_TUNE_SIGMA_N_FLOOR,
                    str(SIGMA_N_FLOOR_FACTORS),
                    AUTO_TUNE_ROUGHNESS_WEIGHT,
                    REG_SCALE_FLOOR_QUANTILE,
                    RAW_LINE_WIDTH,
                    MODEL_LINE_WIDTH,
                    INPUT_POINT_SIZE,
                    INPUT_POINT_EDGE_WIDTH,
                    NYQUIST_ERR_LINEWIDTH,
                    NYQUIST_ERR_CAPSIZE,
                    CI_ALPHA,
                    TITLE_FONT_SIZE,
                ],
            }
        ).to_excel(writer, sheet_name="config", index=False)
        for sample in sheets:
            try:
                raw = standardize_impedance_frame(pd.read_excel(xls, sheet_name=sample))
            except Exception as exc:
                message = f"Failed to read sheet: {exc}"
                log_rows.append({"sample": sample, "condition": "ALL", "branch": "ALL", "success": False, "message": message})
                print(f"Skipped: {sample} | {message}")
                if SKIP_INVALID_SHEETS:
                    continue
                raise
            sample_plot_rows: list[dict] = []
            for condition in conditions:
                try:
                    observed = apply_condition(raw, condition, sample)
                    branch_stats: dict[str, pd.DataFrame] = {}
                    branch_tables: dict[str, pd.DataFrame] = {}
                    for branch in RUN_BRANCHES:
                        stats, log = run_dct_gpht_branch(sample, condition.name, observed, branch)
                        table = prefixed_branch_frame(branch, stats, log)
                        branch_stats[branch] = stats
                        branch_tables[branch] = table
                        log_rows.append(log)
                    out_df = assemble_sheet(raw, observed, branch_tables, condition.name)
                    sheet_name = safe_sheet_name(f"{sample}_{condition.name}", used_sheet_names)
                    out_df.to_excel(writer, sheet_name=sheet_name, index=False)
                    sample_plot_rows.append({"condition": condition.name, "observed": observed, "branch_stats": branch_stats})
                    print(f"Completed: {sample} | {condition.name}")
                except Exception as exc:
                    log_rows.append({"sample": sample, "condition": condition.name, "branch": "ALL", "success": False, "message": repr(exc)})
                    print(f"Failed: {sample} | {condition.name}: {exc}")
            if sample_plot_rows:
                fig_path = plot_sample_grid(sample, raw, sample_plot_rows, used_figure_stems)
                fig_rows.append(
                    {
                        "sample": sample,
                        "condition": "ALL",
                        "figure_stem": fig_path.stem,
                        "figure_png": str(fig_path),
                    }
                )
        log_df = pd.DataFrame(log_rows)
        fig_df = pd.DataFrame(fig_rows)
        log_df.to_excel(writer, sheet_name="analysis_log", index=False)
        fig_df.to_excel(writer, sheet_name="figure_index", index=False)
    print("Data saved to:", output_xlsx)
    print("Figures saved to:", FIG_DIR)
    return output_xlsx, FIG_DIR, log_df, fig_df


### Cell 14 - Experimental Parameter Configuration

This cell configures the input file, output directory, noise levels, sparse-sampling ratios, frequency-limitation/truncation settings, prediction-frequency range, and random seed.

- Purpose: DCT-GPHT unbounded processing of experimental impedance data, dual-branch export, comparison of four condition types, and result saving.
- Main parameters: RUN_FULL_ANALYSIS.These variables control input/output paths, experimental conditions, frequency ranges, or plotting behavior.

In [ ]:
# Run the full workflow; set RUN_FULL_ANALYSIS to False if you only want to inspect or debug earlier cells.
RUN_FULL_ANALYSIS = True
if RUN_FULL_ANALYSIS:
    output_xlsx, fig_dir, log_df, fig_df = run_all()
    display(log_df)
    display(fig_df.head())
else:
    print("Full analysis was skipped. Set RUN_FULL_ANALYSIS to True to generate Excel tables and figures.")


### Cell 15 - Read or Write Tabular Data

This cell reads input workbooks or writes computed results to Excel for later plotting, statistics, and checking.

- Purpose: DCT-GPHT unbounded processing of experimental impedance data, dual-branch export, comparison of four condition types, and result saving.

In [ ]:
# Quick check after the full run.
if "output_xlsx" in globals() and Path(output_xlsx).exists():
    out_xls = pd.ExcelFile(output_xlsx)
    data_sheets = [s for s in out_xls.sheet_names if s not in {"config", "analysis_log", "figure_index"}]
    log_check = pd.read_excel(output_xlsx, sheet_name="analysis_log")
    fig_check = pd.read_excel(output_xlsx, sheet_name="figure_index")
    png_paths = [Path(p) for p in fig_check["figure_png"].dropna()] if "figure_png" in fig_check else []
    existing_png = sum(path.exists() for path in png_paths)
    png_count = len(list(Path(fig_dir).glob("*.png")))
    duplicate_names = fig_check["figure_stem"].duplicated().sum() if "figure_stem" in fig_check else 0
    print("Result workbook:", output_xlsx)
    print("Data directory:", DATA_DIR)
    print("Figure directory:", fig_dir)
    print("Total sheets:", len(out_xls.sheet_names), "| data sheets:", len(data_sheets))
    print("Number of figure_index records:", len(fig_check), "| existing PNG files:", existing_png, "| PNG files in directory:", png_count)
    print("Number of duplicated figure stems:", int(duplicate_names))
    print("Number of failed records:", int((log_check.get("success") == False).sum()) if "success" in log_check else "n/a")
    if data_sheets:
        display(pd.read_excel(output_xlsx, sheet_name=data_sheets[0], nrows=5))
else:
    print("No result workbook has been generated in the current notebook session.")
